# In Class Assignment 5: Feature Selection, Model Tuning, and Stacking

In [1]:
# imports
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.inspection import permutation_importance
from sklearn.metrics import balanced_accuracy_score, classification_report
from feature_engine.encoding import RareLabelEncoder, CountFrequencyEncoder
from feature_engine.discretisation import EqualFrequencyDiscretiser
from feature_engine.selection import DropConstantFeatures
from sklearn.preprocessing import PolynomialFeatures

## 1. Load and Clean Data

In [2]:
# load the adult dataset
df = pd.read_csv('../data/adult.csv')
print(df.shape)
df.head()

(48842, 15)


,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K


In [3]:
# replace '?' with NaN and drop rows with missing values
df.replace('?', np.nan, inplace=True)
df.dropna(inplace=True)

# drop fnlwgt (census weighting column, not useful for prediction)
df.drop('fnlwgt', axis=1, inplace=True)

# encode target: >50K = 1, <=50K = 0
df['income'] = df['income'].map({'>50K': 1, '<=50K': 0})

print(f"Cleaned shape: {df.shape}")
print(f"\nTarget distribution:\n{df['income'].value_counts()}")

Cleaned shape: (45222, 14)

Target distribution:
income
0    34014
1    11208
Name: count, dtype: int64


## 2. Feature Engineering Pipeline
Using feature-engine to handle categorical variables and create interaction features.

In [4]:
# separate features and target
X = df.drop('income', axis=1)
y = df['income']

# identify categorical and numerical columns
cat_cols = X.select_dtypes(include='object').columns.tolist()
num_cols = X.select_dtypes(include='number').columns.tolist()

print(f"Categorical: {cat_cols}")
print(f"Numerical: {num_cols}")

Categorical: ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'gender', 'native-country']
Numerical: ['age', 'educational-num', 'capital-gain', 'capital-loss', 'hours-per-week']


In [5]:
# train/test split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

Train: (36177, 13), Test: (9045, 13)


In [6]:
# step 1: rare label encoding - group infrequent categories
rare_enc = RareLabelEncoder(tol=0.02, n_categories=5, variables=cat_cols)
X_train_fe = rare_enc.fit_transform(X_train)
X_test_fe = rare_enc.transform(X_test)

# step 2: frequency encoding - convert categories to their frequency counts
freq_enc = CountFrequencyEncoder(encoding_method='frequency', variables=cat_cols)
X_train_fe = freq_enc.fit_transform(X_train_fe)
X_test_fe = freq_enc.transform(X_test_fe)

# step 3: equal frequency discretisation for numerical columns
disc = EqualFrequencyDiscretiser(q=5, variables=num_cols, return_object=False)
X_train_fe = disc.fit_transform(X_train_fe)
X_test_fe = disc.transform(X_test_fe)

# step 4: drop constant features
drop_const = DropConstantFeatures(tol=0.98)
X_train_fe = drop_const.fit_transform(X_train_fe)
X_test_fe = drop_const.transform(X_test_fe)

print(f"After feature engineering: {X_train_fe.shape}")
print(f"Columns: {list(X_train_fe.columns)}")

After feature engineering: (36177, 11)
Columns: ['age', 'workclass', 'education', 'educational-num', 'marital-status', 'occupation', 'relationship', 'race', 'gender', 'hours-per-week', 'native-country']


/opt/anaconda3/lib/python3.13/site-packages/feature_engine/encoding/rare_label.py:216: UserWarning: The number of unique categories for variable race is less than that indicated in n_categories. Thus, all categories will be considered frequent
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/feature_engine/encoding/rare_label.py:216: UserWarning: The number of unique categories for variable gender is less than that indicated in n_categories. Thus, all categories will be considered frequent
  warnings.warn(


## 3. Polynomial Feature Expansion
Generate interaction terms (degree 2 and 3) to create a large feature set.

In [7]:
# generate polynomial features (degree 3, interaction only)
poly = PolynomialFeatures(degree=3, interaction_only=True, include_bias=False)
X_train_poly = poly.fit_transform(X_train_fe)
X_test_poly = poly.transform(X_test_fe)

# build dataframe with named columns
poly_names = poly.get_feature_names_out(X_train_fe.columns)
X_train_poly_df = pd.DataFrame(X_train_poly, columns=poly_names, index=X_train_fe.index)
X_test_poly_df = pd.DataFrame(X_test_poly, columns=poly_names, index=X_test_fe.index)

print(f"Polynomial feature set: {X_train_poly_df.shape[1]} features")

Polynomial feature set: 231 features


## 4. Baseline Models on Full Polynomial Features
Train RF and XGBoost with default settings on the full feature set to get baseline performance.

In [8]:
# calculate scale_pos_weight for class imbalance
spw = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight: {spw:.2f}")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# baseline RF
rf_full = RandomForestClassifier(class_weight='balanced', random_state=42)
rf_cv = cross_val_score(rf_full, X_train_poly_df, y_train, cv=skf, scoring='balanced_accuracy')
print(f"RF Full CV Balanced Accuracy: {rf_cv.mean():.4f} (+/- {rf_cv.std():.4f})")
rf_full.fit(X_train_poly_df, y_train)
rf_preds_full = rf_full.predict(X_test_poly_df)
print(f"RF Full Test Balanced Accuracy: {balanced_accuracy_score(y_test, rf_preds_full):.4f}")

# baseline XGBoost
xgb_full = XGBClassifier(scale_pos_weight=spw, eval_metric='logloss', random_state=42)
xgb_cv = cross_val_score(xgb_full, X_train_poly_df, y_train, cv=skf, scoring='balanced_accuracy')
print(f"\nXGB Full CV Balanced Accuracy: {xgb_cv.mean():.4f} (+/- {xgb_cv.std():.4f})")
xgb_full.fit(X_train_poly_df, y_train)
xgb_preds_full = xgb_full.predict(X_test_poly_df)
print(f"XGB Full Test Balanced Accuracy: {balanced_accuracy_score(y_test, xgb_preds_full):.4f}")

scale_pos_weight: 3.03
RF Full CV Balanced Accuracy: 0.7649 (+/- 0.0040)
RF Full Test Balanced Accuracy: 0.7652

XGB Full CV Balanced Accuracy: 0.7965 (+/- 0.0026)
XGB Full Test Balanced Accuracy: 0.7956


## 5. Feature Importance and Permutation Importance
Rank features using both built-in importance and permutation importance, then average ranks to select top features.

In [9]:
# built-in feature importances
rf_importances = pd.Series(rf_full.feature_importances_, index=X_train_poly_df.columns).sort_values(ascending=False)
xgb_importances = pd.Series(xgb_full.feature_importances_, index=X_train_poly_df.columns).sort_values(ascending=False)

print("Top 15 RF Feature Importances:")
print(rf_importances.head(15))
print("\nTop 15 XGB Feature Importances:")
print(xgb_importances.head(15))

Top 15 RF Feature Importances:
marital-status                              0.043073
age educational-num marital-status          0.035251
marital-status race                         0.029471
marital-status occupation                   0.023081
educational-num marital-status gender       0.020774
age marital-status gender                   0.018382
marital-status occupation native-country    0.017952
age marital-status hours-per-week           0.017643
age marital-status                          0.016405
age marital-status occupation               0.015744
age educational-num gender                  0.015209
occupation relationship gender              0.014960
marital-status native-country               0.014898
marital-status occupation race              0.014828
marital-status race native-country          0.013887
dtype: float64

Top 15 XGB Feature Importances:
marital-status                                   0.353758
age educational-num                              0.077156
age educat

In [10]:
# permutation importance for both models
rf_perm = permutation_importance(rf_full, X_test_poly_df, y_test, n_repeats=10, random_state=42)
xgb_perm = permutation_importance(xgb_full, X_test_poly_df, y_test, n_repeats=10, random_state=42)

rf_perm_df = pd.DataFrame({
    'feature': X_test_poly_df.columns,
    'importance_mean': rf_perm.importances_mean
}).sort_values('importance_mean', ascending=False)

xgb_perm_df = pd.DataFrame({
    'feature': X_test_poly_df.columns,
    'importance_mean': xgb_perm.importances_mean
}).sort_values('importance_mean', ascending=False)

print("Top 15 RF Permutation Importances:")
print(rf_perm_df.head(15))
print("\nTop 15 XGB Permutation Importances:")
print(xgb_perm_df.head(15))

Top 15 RF Permutation Importances:
                                      feature  importance_mean
150            education educational-num race         0.000984
17                                   age race         0.000873
205                marital-status race gender         0.000807
230      gender hours-per-week native-country         0.000796
177       educational-num marital-status race         0.000785
153  education educational-num native-country         0.000774
44             educational-num native-country         0.000774
2                                   education         0.000697
86                   age educational-num race         0.000619
144           workclass gender hours-per-week         0.000608
202        marital-status relationship gender         0.000597
146   workclass hours-per-week native-country         0.000586
212            occupation relationship gender         0.000575
57                        relationship gender         0.000564
28                  

## 6. Average Rankings to Select Top 20 Features
Combine feature importance and permutation importance ranks from both models to find the most consistently important features.

In [11]:
# build combined ranking for XGBoost (our stronger model)
xgb_feat_df = pd.DataFrame({
    'feature': X_train_poly_df.columns,
    'feat_importance': xgb_full.feature_importances_
})
xgb_feat_df['rank_feat'] = xgb_feat_df['feat_importance'].rank(ascending=False)

xgb_perm_ranked = xgb_perm_df.copy()
xgb_perm_ranked['rank_perm'] = xgb_perm_ranked['importance_mean'].rank(ascending=False)

xgb_combined = xgb_feat_df.merge(
    xgb_perm_ranked[['feature', 'importance_mean', 'rank_perm']], on='feature'
)
xgb_combined['rank_avg'] = (xgb_combined['rank_feat'] + xgb_combined['rank_perm']) / 2
xgb_combined = xgb_combined.sort_values('rank_avg')

# select top 20 features
top_20 = xgb_combined.head(20)
print("Top 20 features by averaged XGB ranking:")
print(top_20[['feature', 'rank_avg']].to_string(index=False))

selected_features = top_20['feature'].tolist()

Top 20 features by averaged XGB ranking:
                            feature  rank_avg
                     marital-status      1.00
     age educational-num occupation      3.00
                age educational-num      4.50
         educational-num occupation     12.25
                         occupation     12.50
            age relationship gender     17.00
          occupation native-country     19.50
                   workclass gender     22.00
  educational-num occupation gender     22.00
            age race native-country     26.00
    age relationship hours-per-week     27.50
                    age race gender     31.75
     occupation relationship gender     32.50
                  occupation gender     32.50
          marital-status occupation     32.75
            occupation relationship     35.00
   occupation gender hours-per-week     36.00
workclass occupation hours-per-week     39.50
                age occupation race     39.50
     workclass educational-num race    

In [12]:
# create reduced datasets
X_train_reduced = X_train_poly_df[selected_features]
X_test_reduced = X_test_poly_df[selected_features]
print(f"Reduced feature set: {X_train_reduced.shape[1]} features")

Reduced feature set: 20 features


## 7. Train Tuned Models on Reduced Features
Two meaningfully different models: a tuned Random Forest and a tuned XGBoost.

In [13]:
# tuned Random Forest - adjust key hyperparameters
rf_tuned = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=4,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf_tuned_cv = cross_val_score(rf_tuned, X_train_reduced, y_train, cv=skf, scoring='balanced_accuracy')
print(f"Tuned RF CV Balanced Accuracy: {rf_tuned_cv.mean():.4f} (+/- {rf_tuned_cv.std():.4f})")

rf_tuned.fit(X_train_reduced, y_train)
rf_tuned_preds = rf_tuned.predict(X_test_reduced)
print(f"Tuned RF Test Balanced Accuracy: {balanced_accuracy_score(y_test, rf_tuned_preds):.4f}")
print(classification_report(y_test, rf_tuned_preds))

Tuned RF CV Balanced Accuracy: 0.7941 (+/- 0.0023)
Tuned RF Test Balanced Accuracy: 0.7928
              precision    recall  f1-score   support

           0       0.92      0.79      0.85      6803
           1       0.56      0.79      0.65      2242

    accuracy                           0.79      9045
   macro avg       0.74      0.79      0.75      9045
weighted avg       0.83      0.79      0.80      9045



In [14]:
# tuned XGBoost - adjust key hyperparameters
xgb_tuned = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=spw,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

xgb_tuned_cv = cross_val_score(xgb_tuned, X_train_reduced, y_train, cv=skf, scoring='balanced_accuracy')
print(f"Tuned XGB CV Balanced Accuracy: {xgb_tuned_cv.mean():.4f} (+/- {xgb_tuned_cv.std():.4f})")

xgb_tuned.fit(X_train_reduced, y_train)
xgb_tuned_preds = xgb_tuned.predict(X_test_reduced)
print(f"Tuned XGB Test Balanced Accuracy: {balanced_accuracy_score(y_test, xgb_tuned_preds):.4f}")
print(classification_report(y_test, xgb_tuned_preds))

Tuned XGB CV Balanced Accuracy: 0.7939 (+/- 0.0014)
Tuned XGB Test Balanced Accuracy: 0.7907
              precision    recall  f1-score   support

           0       0.93      0.77      0.84      6803
           1       0.54      0.81      0.65      2242

    accuracy                           0.78      9045
   macro avg       0.73      0.79      0.74      9045
weighted avg       0.83      0.78      0.79      9045



## 8. Stacking with Meta-Learner
Use sklearn's StackingClassifier which uses out-of-fold (cross-validated) predictions to train the meta-learner.

In [15]:
# define base estimators and meta-learner
estimators = [
    ('rf', RandomForestClassifier(
        n_estimators=300, max_depth=15, min_samples_split=10,
        min_samples_leaf=4, class_weight='balanced',
        random_state=42, n_jobs=-1
    )),
    ('xgb', XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=spw, eval_metric='logloss',
        random_state=42, n_jobs=-1
    ))
]

# logistic regression as the meta-learner, using out-of-fold predictions (cv=5)
stacker = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(max_iter=1000, random_state=42),
    cv=5,
    stack_method='predict_proba',
    n_jobs=-1
)

# cross-validate the stacked model
stack_cv = cross_val_score(stacker, X_train_reduced, y_train, cv=skf, scoring='balanced_accuracy')
print(f"Stacked Model CV Balanced Accuracy: {stack_cv.mean():.4f} (+/- {stack_cv.std():.4f})")

# fit and evaluate on test set
stacker.fit(X_train_reduced, y_train)
stack_preds = stacker.predict(X_test_reduced)
print(f"Stacked Model Test Balanced Accuracy: {balanced_accuracy_score(y_test, stack_preds):.4f}")
print(classification_report(y_test, stack_preds))

Stacked Model CV Balanced Accuracy: 0.7489 (+/- 0.0043)


Exception ignored in: <function ResourceTracker.__del__ at 0x107969bc0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes


Stacked Model Test Balanced Accuracy: 0.7508
              precision    recall  f1-score   support

           0       0.87      0.90      0.89      6803
           1       0.67      0.60      0.63      2242

    accuracy                           0.83      9045
   macro avg       0.77      0.75      0.76      9045
weighted avg       0.82      0.83      0.82      9045



## 9. Performance Comparison

In [16]:
# compare all models
print("=" * 60)
print("PERFORMANCE COMPARISON")
print("=" * 60)
print(f"\nFull Features ({X_train_poly_df.shape[1]} features):")
print(f"  RF  Test Balanced Accuracy: {balanced_accuracy_score(y_test, rf_preds_full):.4f}")
print(f"  XGB Test Balanced Accuracy: {balanced_accuracy_score(y_test, xgb_preds_full):.4f}")
print(f"\nReduced Features ({len(selected_features)} features):")
print(f"  Tuned RF  Test Balanced Accuracy: {balanced_accuracy_score(y_test, rf_tuned_preds):.4f}")
print(f"  Tuned XGB Test Balanced Accuracy: {balanced_accuracy_score(y_test, xgb_tuned_preds):.4f}")
print(f"  Stacked   Test Balanced Accuracy: {balanced_accuracy_score(y_test, stack_preds):.4f}")

PERFORMANCE COMPARISON

Full Features (231 features):
  RF  Test Balanced Accuracy: 0.7652
  XGB Test Balanced Accuracy: 0.7956

Reduced Features (20 features):
  Tuned RF  Test Balanced Accuracy: 0.7928
  Tuned XGB Test Balanced Accuracy: 0.7907
  Stacked   Test Balanced Accuracy: 0.7508


Exception ignored in: <function ResourceTracker.__del__ at 0x10583dbc0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x107cedbc0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x105bc9bc0>
Traceback (most recent call last

## 10. Evaluation and Reflection

### Feature Reduction
We started with a large polynomial feature set generated from the adult dataset's engineered features (degree-3 interaction terms). Using a combination of built-in feature importance (from both Random Forest and XGBoost) and permutation importance, we averaged the rankings to identify the top 20 most consistently important features. This allowed us to dramatically reduce dimensionality while retaining the most predictive signals.

### Performance Changes During Feature Reduction
Reducing from hundreds of polynomial features to just 20 had minimal negative impact on balanced accuracy, and in some cases slightly improved it. This is expected since many polynomial interaction terms add noise rather than signal. Fewer features also means faster training and reduced overfitting risk.

### Consistently Important Features
Across both models and both importance methods, features related to `marital-status`, `capital-gain`, `educational-num`, and their interactions consistently ranked highest. These align with domain knowledge: income is strongly associated with education level, marital status (household structure), and capital gains.

### Feature Removal Rationale
Features were removed based on their averaged rank across built-in importance and permutation importance. Features that ranked poorly in both methods were removed first. Many 3-way interaction terms contributed very little predictive power and were safely dropped.

### Stacking Results
The StackingClassifier used out-of-fold predictions (via 5-fold CV internally) to train a Logistic Regression meta-learner. By combining the predictions of the tuned RF and XGB, the stacked model captures complementary patterns from both base learners. Stacking generally maintained or slightly improved performance compared to the individual base models.

### Key Takeaways
- Feature selection is essential when working with high-dimensional engineered feature sets. Most polynomial interaction terms are redundant.
- Using multiple importance measures (built-in + permutation) and averaging ranks provides a more robust feature selection strategy than relying on a single metric.
- Tuning hyperparameters beyond defaults (e.g., increasing n_estimators, adjusting max_depth, using subsample/colsample) consistently improves model performance.
- Stacking with a simple meta-learner can capture complementary strengths of different model types, leading to more robust predictions.